# Libraries

In [2]:
!pip -q install "lm_eval[hf]"
!pip -q install compressed-tensors
!pip install --upgrade transformers #FIX

  Using cached transformers-5.4.0-py3-none-any.whl.metadata (32 kB)
  Using cached huggingface_hub-1.8.0-py3-none-any.whl.metadata (13 kB)
Using cached transformers-5.4.0-py3-none-any.whl (10.1 MB)
Using cached huggingface_hub-1.8.0-py3-none-any.whl (625 kB)
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 0.36.2
    Uninstalling huggingface_hub-0.36.2:
      Successfully uninstalled huggingface_hub-0.36.2
  Attempting uninstall: transformers
    Found existing installation: transformers 4.57.6
    Uninstalling transformers-4.57.6:
      Successfully uninstalled transformers-4.57.6
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
compressed-tensors 0.14.0.1 requires transformers<5.0.0, but you have transformers 5.4.0 which is incompatible.


In [3]:
# 2. Libraries
import random
import numpy as np
import torch
import os
import gc

import lm_eval
import subprocess

from huggingface_hub import login
from kaggle_secrets import UserSecretsClient

import shutil
from IPython.display import FileLink

# Functions

In [4]:
# --- 2. REPRODUCIBILITY ---
def set_reproducibility(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    # Ensure deterministic behavior in CuDNN
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    print(f"Global seed set to {seed}")

# Global Variables

In [10]:
model_name = "EdgeCompress01/Qwen3-4B-SparseGPT"
subfolder = "Models/50"
SEED = 42
os.environ["HF_ALLOW_CODE_EVAL"] = "1"

# Seed for reproducibility

In [11]:
set_reproducibility(SEED)

Global seed set to 42


# Login to huggingface

In [12]:
# --- 3. HUGGING FACE AUTHENTICATION ---
user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN")
login(token=hf_token)
print("Logging into Hugging Face...")

Logging into Hugging Face...


# Do Evaluation

**Configurations**

In [14]:
tasks_config = {
    "gsm8k": {
        "task_name": "gsm8k_cot",
        "fewshot": 2   # 🔻 reduced from 8
    },
    "mmlu": {
        "task_name": "mmlu",
        "fewshot": 2   # 🔻 reduced from 5
    },
    "arc_challenge": {
        "task_name": "arc_challenge",
        "fewshot": 0   # keep
    },
    "wikitext": {
        "task_name": "wikitext",
        "fewshot": 0   # keep
    }
}

In [15]:
for task, cfg in tasks_config.items():
    print(f"Starting evaluation for: {task}")

    model_args = f"pretrained={model_name},subfolder={subfolder},max_length=2048,dtype=float16,parallelize=True"
    command = [
        "lm_eval",
        "--model", "hf",
        "--model_args", model_args,
        "--tasks", cfg["task_name"],
        "--batch_size", str(3),   
        "--verbosity", "WARNING",
        "--seed", str(SEED),
        "--limit", str(100),
        "--num_fewshot", str(cfg["fewshot"]),
        "--output_path", f"evaluation_results/{task}"
    ]

    subprocess.run(command, check=True)
    
    torch.cuda.empty_cache()
    gc.collect()
    print(f"Finished {task}\n")

Starting evaluation for: gsm8k


2026-03-27:12:57:45 WARNING  [config.evaluate_config:281] --limit SHOULD ONLY BE USED FOR TESTING. REAL METRICS SHOULD NOT BE COMPUTED USING LIMIT.
2026-03-27:12:57:51 INFO     [_cli.run:376] Selected Tasks: ['gsm8k_cot']
Generating test split: 100%|██████████| 1319/1319 [00:00<00:00, 288937.53 examples/s]
2026-03-27:12:58:44 WARNING  [evaluator:333] Overwriting default num_fewshot of gsm8k_cot from 8 to 2
Running generate_until requests: 100%|██████████| 100/100 [08:50<00:00,  5.31s/it]
fatal: not a git repository (or any parent up to mount point /kaggle)
Stopping at filesystem boundary (GIT_DISCOVERY_ACROSS_FILESYSTEM not set).


hf ({'pretrained': 'EdgeCompress01/Qwen3-4B-SparseGPT', 'subfolder': 'Models/50', 'max_length': 2048, 'dtype': 'float16', 'parallelize': True}), gen_kwargs: ({}), limit: 100.0, num_fewshot: 2, batch_size: 3
|  Tasks  |Version|     Filter     |n-shot|  Metric   |   |Value|   |Stderr|
|---------|------:|----------------|-----:|-----------|---|----:|---|-----:|
|gsm8k_cot|      3|flexible-extract|     2|exact_match|↑  | 0.18|±  |0.0386|
|         |       |strict-match    |     2|exact_match|↑  | 0.10|±  |0.0302|

Finished gsm8k

Starting evaluation for: mmlu


2026-03-27:13:07:40 WARNING  [config.evaluate_config:281] --limit SHOULD ONLY BE USED FOR TESTING. REAL METRICS SHOULD NOT BE COMPUTED USING LIMIT.
2026-03-27:13:07:45 INFO     [_cli.run:376] Selected Tasks: ['mmlu']
Generating dev split: 100%|██████████| 5/5 [00:00<00:00, 2757.60 examples/s]
2026-03-27:13:09:23 WARNING  [evaluator:333] Overwriting default num_fewshot of mmlu_abstract_algebra from None to 2
2026-03-27:13:09:23 WARNING  [evaluator:333] Overwriting default num_fewshot of mmlu_anatomy from None to 2
2026-03-27:13:09:23 WARNING  [evaluator:333] Overwriting default num_fewshot of mmlu_astronomy from None to 2
2026-03-27:13:09:23 WARNING  [evaluator:333] Overwriting default num_fewshot of mmlu_college_biology from None to 2
2026-03-27:13:09:23 WARNING  [evaluator:333] Overwriting default num_fewshot of mmlu_college_chemistry from None to 2
2026-03-27:13:09:23 WARNING  [evaluator:333] Overwriting default num_fewshot of mmlu_college_computer_science from None to 2
2026-03-27:1

hf ({'pretrained': 'EdgeCompress01/Qwen3-4B-SparseGPT', 'subfolder': 'Models/50', 'max_length': 2048, 'dtype': 'float16', 'parallelize': True}), gen_kwargs: ({}), limit: 100.0, num_fewshot: 2, batch_size: 3
|                 Tasks                 |Version|Filter|n-shot|Metric|   |Value |   |Stderr|
|---------------------------------------|------:|------|-----:|------|---|-----:|---|-----:|
|mmlu                                   |      2|none  |      |acc   |↑  |0.5295|±  |0.0064|
| - humanities                          |      2|none  |      |acc   |↑  |0.5115|±  |0.0134|
|  - formal_logic                       |      1|none  |     2|acc   |↑  |0.4100|±  |0.0494|
|  - high_school_european_history       |      1|none  |     2|acc   |↑  |0.6700|±  |0.0473|
|  - high_school_us_history             |      1|none  |     2|acc   |↑  |0.6000|±  |0.0492|
|  - high_school_world_history          |      1|none  |     2|acc   |↑  |0.6800|±  |0.0469|
|  - international_law         Finished mmlu

Sta

2026-03-27:13:29:40 WARNING  [config.evaluate_config:281] --limit SHOULD ONLY BE USED FOR TESTING. REAL METRICS SHOULD NOT BE COMPUTED USING LIMIT.
2026-03-27:13:29:45 INFO     [_cli.run:376] Selected Tasks: ['arc_challenge']
Generating validation split: 100%|██████████| 299/299 [00:00<00:00, 106640.89 examples/s]
2026-03-27:13:29:59 WARNING  [evaluator:333] Overwriting default num_fewshot of arc_challenge from None to 0
Running loglikelihood requests: 100%|██████████| 400/400 [00:12<00:00, 32.55it/s]
fatal: not a git repository (or any parent up to mount point /kaggle)
Stopping at filesystem boundary (GIT_DISCOVERY_ACROSS_FILESYSTEM not set).


Finished arc_challenge

Starting evaluation for: wikitext


2026-03-27:13:30:17 WARNING  [config.evaluate_config:281] --limit SHOULD ONLY BE USED FOR TESTING. REAL METRICS SHOULD NOT BE COMPUTED USING LIMIT.
2026-03-27:13:30:22 INFO     [_cli.run:376] Selected Tasks: ['wikitext']
Loading weights: 100%|██████████| 398/398 [00:02<00:00, 185.66it/s]
2026-03-27:13:30:34 WARNING  [api.task:729] [Task: wikitext] metric word_perplexity is defined, but aggregation is not. using default aggregation=weighted_perplexity
2026-03-27:13:30:34 WARNING  [api.task:741] [Task: wikitext] metric word_perplexity is defined, but higher_is_better is not. using default higher_is_better=False
2026-03-27:13:30:34 WARNING  [api.task:729] [Task: wikitext] metric byte_perplexity is defined, but aggregation is not. using default aggregation=weighted_perplexity
2026-03-27:13:30:34 WARNING  [api.task:741] [Task: wikitext] metric byte_perplexity is defined, but higher_is_better is not. using default higher_is_better=False
2026-03-27:13:30:34 WARNING  [api.task:729] [Task: wiki

Finished wikitext



# Save reports

In [16]:
zip_path = shutil.make_archive(f"evaluation_results", 'zip', f"evaluation_results")
FileLink(zip_path)

/kaggle/working/evaluation_results.zip